<a href="https://colab.research.google.com/github/SynPmk/Cats_vs_Dogs_classification/blob/main/gsadpciris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


from sklearn.datasets import load_iris
import pandas as pd


iris = load_iris()


X = iris.data


y = iris.target


df = pd.DataFrame(
    X,
    columns=iris.feature_names
)


df["target"] = y


print("Dataset shape:", X.shape)
print("\nIris Dataset:")
display(df)


print("\nClass names:")
print(iris.target_names)

import numpy as np
import pandas as pd


N = X.shape[0]   # 150
l = X.shape[1]   # 4

D = np.zeros((N, N))


for i in range(N):
    for j in range(N):
        D[i, j] = np.sqrt(np.sum((X[i, :] - X[j, :]) ** 2))


D_df = pd.DataFrame(
    D,
    index=range(1, N + 1),
    columns=range(1, N + 1)
)


display(D_df)


D_flat = D.flatten()


D_sorted = np.sort(D_flat)


print("Number of distances:", len(D_sorted))
print("Sorted distances:")

for i, value in enumerate(D_sorted, start=1):
    print(f"{i}: {value}")

import numpy as np

N = X.shape[0]


rho = np.zeros(N)

for i in range(N):
    for j in range(N):
        rho[i] += np.exp(-(D[i, j] ** 2) / (dc ** 2))

highest_density_index = np.argmax(rho)

highest_density_point = highest_density_index + 1

highest_density = rho[highest_density_index]


delta_highest = np.max(D[highest_density_index, :])



print("Number of data points:", N)
print("dc =", dc)

print("\nLocal densities (rho):")
for i in range(N):
    print(f"Data point {i+1}: rho = {rho[i]:.15f}")

print("\nHighest-density data point:")
print("Data point =", highest_density_point)
print("rho =", highest_density)

print("\nDelta for highest-density point:")
print("delta =", delta_highest)


delta = np.zeros(N)

for i in range(N):
    higher_density_points = np.where(rho > rho[i])[0]

    if len(higher_density_points) > 0:
        distances = D[i, higher_density_points]
        delta[i] = np.min(distances)

    else:
        delta[i] = np.max(D[i, :])

print("Delta values for all 150 data points:\n")

for i in range(N):
    print(f"Data point {i+1}: rho = {rho[i]:.15f}, delta = {delta[i]:.15f}")

gamma = rho * delta

sorted_indices = np.argsort(gamma)[::-1]

gamma_sorted = gamma[sorted_indices]
rho_sorted = rho[sorted_indices]
delta_sorted = delta[sorted_indices]

print("Points sorted by gamma in descending order:\n")

for rank, idx in enumerate(sorted_indices, start=1):
    print(
        f"Rank {rank:3d} | "
        f"Data point {idx + 1:3d} | "
        f"rho = {rho[idx]:.15f} | "
        f"delta = {delta[idx]:.15f} | "
        f"gamma = {gamma[idx]:.15f}"

K = 3
candidate_indices = sorted_indices[:K]
C = X[candidate_indices, :]

print("K =", K)
print("\nCandidate cluster center indices (1-based):")

for i, idx in enumerate(candidate_indices, start=1):
    print(f"Center {i}: Data point O_{idx + 1}")

print("\nCandidate cluster center set C:")
print(C)

s = 50                  # Population size
G0 = 100                # Initial gravitational constant
alpha = 20              # Paper parameter
max_iter = 500
epsilon = 1e-12

np.random.seed(42)

X_population = np.zeros((s, K, l))

# X1 <- C
X_population[0] = C.copy()

# Xi <- C * rd
for i in range(1, s):

    rd = np.random.uniform(0, 1)

    while rd == 0:
        rd = np.random.uniform(0, 1)

    X_population[i] = C * rd

V = np.zeros((s, K, l))

global_best_fitness = np.inf
global_best_position = None
global_best_assignment = None

for iteration in range(max_iter):

    distance = np.zeros((s, N, K))

    for t in range(s):

        for i in range(N):

            for j in range(K):

                distance[t, i, j] = np.sqrt(
                    np.sum(
                        (X[i] - X_population[t, j]) ** 2
                    )
                )

    IDX = np.argmin(distance, axis=2)

    fitness = np.zeros(s)

    for t in range(s):

        for i in range(N):

            cluster = IDX[t, i]

            fitness[t] += (
                distance[t, i, cluster] ** 2

    current_best_index = np.argmin(fitness)

    if fitness[current_best_index] < global_best_fitness:

        global_best_fitness = fitness[current_best_index]

        global_best_position = (
            X_population[current_best_index].copy()
        )

        global_best_assignment = (
            IDX[current_best_index].copy()
        )

    best = np.min(fitness)
    worst = np.max(fitness)

    if abs(best - worst) < epsilon:

        M = np.ones(s) / s

    else:

        m = (
            (fitness - worst)
            / (best - worst)
        )

        M = m / np.sum(m)


    G = G0 * np.exp(-alpha * iteration / max_iter)


    acceleration = np.zeros(
        (s, K, l)
    )

    K0 = s

    Kbest = int(round(
        K0 - ((K0 - 1) * iteration / (max_iter - 1)
    )))

    Kbest_agents = np.argsort(M)[-Kbest:]

    for i in range(s):

        for j in Kbest_agents:

            if i == j:
                continue

            difference = (
                X_population[j]
                - X_population[i]
            )

            R_ij = np.sqrt(
                np.sum(difference ** 2)
            )

            rand_j = np.random.uniform(0, 1)

            acceleration[i] += (
                G
                * rand_j
                * M[j]
                * difference
                / (R_ij + epsilon)
            )

    for i in range(s):

        rand_i = np.random.uniform(0, 1)

        V[i] = (
            rand_i * V[i]
            + acceleration[i]
        )

    X_population = (
        X_population + V
    )

    if (iteration + 1) % 10 == 0:

        print(
            f"Iteration {iteration + 1:3d} "
            f"Best Fitness = "
            f"{global_best_fitness}"
        )

print("\n============================================")
print("FINAL RESULT")
print("============================================")

print("\nBest Fitness:")
print(global_best_fitness)

print("\nBest Cluster Centers:")
print(global_best_position)

print("\nBest Cluster Assignment:")
print(global_best_assignment + 1)

cluster_sizes = np.bincount(
    global_best_assignment,
    minlength=K
)

print("\nCluster Sizes:")

for j in range(K):

    print(
        f"Cluster {j + 1}: "
        f"{cluster_sizes[j]} data points"
    )








from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from scipy.optimize import linear_sum_assignment
import numpy as np

y_true = iris.target
y_pred = global_best_assignment

# ============================================================
# 1. ACCURACY (ACC)
#    Find the best mapping between predicted clusters
#    and the true Iris classes
# ============================================================

contingency = np.zeros((K, K), dtype=int)

for i in range(N):
    contingency[y_pred[i], y_true[i]] += 1

row_ind, col_ind = linear_sum_assignment(-contingency)

ACC = contingency[row_ind, col_ind].sum() / N


# ============================================================
# 2. NMI
#    Geometric normalization:
#    NMI = MI / sqrt(H(X) * H(Y))
# ============================================================

NMI = normalized_mutual_info_score(
    y_true,
    y_pred,
    average_method='geometric'
)


# ============================================================
# 3. ARI
# ============================================================

ARI = adjusted_rand_score(y_true, y_pred)


# ============================================================
# 4. F-MEASURE
#    Pairwise precision / recall
# ============================================================

TP = 0
FP = 0
FN = 0

for i in range(N):
    for j in range(i + 1, N):

        same_true = (y_true[i] == y_true[j])
        same_pred = (y_pred[i] == y_pred[j])

        if same_true and same_pred:
            TP += 1
        elif not same_true and same_pred:
            FP += 1
        elif same_true and not same_pred:
            FN += 1

precision = TP / (TP + FP)
recall = TP / (TP + FN)

F_measure = 2 * precision * recall / (precision + recall)


# ============================================================
# RESULTS
# ============================================================

print("\n============================================")
print("EVALUATION METRICS")
print("============================================")

print(f"ACC         = {ACC:.6f}")
print(f"NMI         = {NMI:.6f}")
print(f"F-measure   = {F_measure:.6f}")
print(f"ARI         = {ARI:.6f}")

Dataset shape: (150, 4)

Iris Dataset:


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2



Class names:
['setosa' 'versicolor' 'virginica']


,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
1,0.000000,0.538516,0.509902,0.648074,0.141421,0.616441,0.519615,0.173205,0.921954,0.469042,...,5.019960,4.636809,4.208325,5.257376,5.136146,4.654031,4.276681,4.459821,4.650806,4.140048
2,0.538516,0.000000,0.300000,0.331662,0.608276,1.090871,0.509902,0.424264,0.509902,0.173205,...,5.072475,4.702127,4.180909,5.320714,5.206726,4.700000,4.249706,4.498889,4.718050,4.153312
3,0.509902,0.300000,0.000000,0.244949,0.509902,1.086278,0.264575,0.412311,0.435890,0.316228,...,5.228767,4.868265,4.334743,5.475400,5.353504,4.864155,4.430576,4.661545,4.848711,4.298837
4,0.648074,0.331662,0.244949,0.000000,0.648074,1.166190,0.331662,0.500000,0.300000,0.316228,...,5.104900,4.760252,4.177320,5.349766,5.232590,4.745524,4.288356,4.533211,4.719110,4.149699
5,0.141421,0.608276,0.509902,0.648074,0.000000,0.616441,0.458258,0.223607,0.921954,0.529150,...,5.061620,4.686150,4.246175,5.297169,5.173007,4.701064,4.330127,4.504442,4.678675,4.173727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,4.654031,4.700000,4.864155,4.745524,4.701064,4.284857,4.796874,4.598913,4.914265,4.666905,...,0.424264,0.244949,1.034408,0.734847,0.616441,0.000000,0.781025,0.360555,0.670820,0.948683
147,4.276681,4.249706,4.430576,4.288356,4.330127,3.988734,4.384062,4.200000,4.429447,4.220190,...,1.063015,0.943398,0.547723,1.307670,1.284523,0.781025,0.000000,0.583095,1.067708,0.655744
148,4.459821,4.498889,4.661545,4.533211,4.504442,4.102438,4.593474,4.397727,4.701064,4.457578,...,0.608276,0.519615,0.774597,0.842615,0.793725,0.360555,0.583095,0.000000,0.616441,0.640312
149,4.650806,4.718050,4.848711,4.719110,4.678675,4.264974,4.749737,4.589118,4.888763,4.672259,...,0.624500,0.818535,0.948683,0.806226,0.624500,0.670820,1.067708,0.616441,0.000000,0.768115


Streaming output truncated to the last 5000 lines.
18059: 4.149698784249286
18060: 4.149698784249286
18061: 4.149698784249286
18062: 4.149698784249286
18063: 4.149698784249286
18064: 4.149698784249286
18065: 4.149698784249286
18066: 4.149698784249286
18067: 4.150903516103452
18068: 4.150903516103452
18069: 4.1521078984053394
18070: 4.1521078984053394
18071: 4.1521078984053394
18072: 4.1521078984053394
18073: 4.153311931459037
18074: 4.153311931459037
18075: 4.153311931459037
18076: 4.153311931459037
18077: 4.153311931459037
18078: 4.153311931459037
18079: 4.153311931459037
18080: 4.153311931459037
18081: 4.154515615568198
18082: 4.154515615568198
18083: 4.154515615568198
18084: 4.154515615568198
18085: 4.154515615568198
18086: 4.154515615568198
18087: 4.154515615568198
18088: 4.154515615568198
18089: 4.155718951036031
18090: 4.155718951036031
18091: 4.158124577258358
18092: 4.158124577258358
18093: 4.158124577258358
18094: 4.158124577258358
18095: 4.158124577258358
18096: 4.15812457725